# 🚀 Lab Module 30 — Capstone: Future of AI Agents

**Module:** 30 — Future of AI Agents
**Thời gian:** ~90 phút
**Nội dung:**

1. **AI Cost Trajectory Simulator** — Project LoanBot economics to 2030
2. **Extended Thinking Demo** — claude-sonnet-4-6 với thinking enabled cho complex borderline cases
3. **Agent Ecosystem Simulator** — LoanBot v5.0 với external agent connections
4. **Career Path Assessment** — Skill gap analysis sau 30 modules
5. **Capstone Integration Exercise** — LoanBot v5.0 Full Pipeline (TODO → Solution)

---
> **Prerequisite:** `pip install anthropic` · API key trong environment: `ANTHROPIC_API_KEY`

## Section 1: AI Cost Trajectory Simulator

Mô phỏng xu hướng cost-per-token từ 2020–2030 và impact lên LoanBot economics.

In [ ]:
from dataclasses import dataclass
from typing import List

@dataclass
class YearlySnapshot:
    year: int
    haiku_cost_per_mtok: float      # USD per million tokens
    sonnet_cost_per_mtok: float
    loanbot_cost_per_case_vnd: float
    break_even_cases_month: int
    new_viable_markets: List[str]

class AICostTrajectorySimulator:
    """
    Simulates AI cost decline and its impact on LoanBot economics.
    Base: 2026 actuals (M29). Applies ~30% annual cost reduction (conservative).
    """

    # 2026 baselines (from M29)
    BASE_YEAR = 2026
    BASE_HAIKU_COST = 0.80       # USD / MTok (blended input+output)
    BASE_SONNET_COST = 9.00      # USD / MTok
    BASE_LOANBOT_COST_VND = 115  # VNĐ/case (weighted avg M29)
    BASE_BREAK_EVEN = 6250       # cases/month (M29)

    # Annual cost reduction rate (conservative estimate)
    ANNUAL_COST_REDUCTION = 0.35  # 35% cheaper each year

    # VNĐ/USD exchange rate
    USD_TO_VND = 25000

    def project(self, target_year: int) -> YearlySnapshot:
        years_ahead = target_year - self.BASE_YEAR
        factor = (1 - self.ANNUAL_COST_REDUCTION) ** years_ahead

        haiku_cost = self.BASE_HAIKU_COST * factor
        sonnet_cost = self.BASE_SONNET_COST * factor
        loanbot_cost = self.BASE_LOANBOT_COST_VND * factor

        # Break-even scales inversely with AI cost reduction
        break_even = max(50, int(self.BASE_BREAK_EVEN * factor))

        # Identify newly viable markets at each cost level
        markets = []
        if loanbot_cost < 100:
            markets.append("Small credit cooperatives (500-2,000 cases/month)")
        if loanbot_cost < 50:
            markets.append("Microfinance organizations (200-500 cases/month)")
        if loanbot_cost < 20:
            markets.append("Rural development banks (<200 cases/month)")
        if loanbot_cost < 10:
            markets.append("Individual lenders / P2P platforms")
        if loanbot_cost < 5:
            markets.append("NGO microloan programs (any scale)")

        return YearlySnapshot(
            year=target_year,
            haiku_cost_per_mtok=round(haiku_cost, 4),
            sonnet_cost_per_mtok=round(sonnet_cost, 4),
            loanbot_cost_per_case_vnd=round(loanbot_cost, 1),
            break_even_cases_month=break_even,
            new_viable_markets=markets
        )

    def run_projection(self):
        sim = AICostTrajectorySimulator()
        print("=" * 70)
        print("LoanBot Cost Trajectory — 2026 → 2030")
        print("=" * 70)
        for year in [2026, 2027, 2028, 2029, 2030]:
            snap = sim.project(year)
            marker = " ← HIỆN TẠI" if year == 2026 else ""
            print(f"\n📅 {snap.year}{marker}")
            print(f"   Haiku cost:      ${snap.haiku_cost_per_mtok:.4f}/MTok")
            print(f"   Sonnet cost:     ${snap.sonnet_cost_per_mtok:.4f}/MTok")
            print(f"   LoanBot avg:     {snap.loanbot_cost_per_case_vnd:.1f} VNĐ/case")
            print(f"   Break-even:      {snap.break_even_cases_month:,} cases/month")
            if snap.new_viable_markets:
                print(f"   🆕 New markets:  {'; '.join(snap.new_viable_markets[:2])}")
        print("\n" + "=" * 70)

AICostTrajectorySimulator().run_projection()

## Section 2: Extended Thinking Demo

claude-sonnet-4-6 với thinking enabled để handle borderline cases phức tạp mà M26 multi-agent debate cần.

In [ ]:
import anthropic
import os

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

# Borderline case: FC-2024-004 (score 645, borderline từ M1)
BORDERLINE_CASE = """
Loan Application FC-2024-004:
- Applicant: Nguyễn Văn D, Nghệ An province
- Requested amount: 180,000,000 VNĐ (personal loan)
- Credit score: 645 (borderline — thresholds: <580 REJECT, 580-659 REVIEW, ≥660 APPROVE)
- Monthly income: 12,500,000 VNĐ (seasonal agricultural work — lower Jan-Mar)
- Employment: Self-employed farmer, 5 years
- DTI: 42% (slightly above 40% preferred ceiling)
- Current loans: 1 active (motorcycle loan, 2M/month, 8 months remaining)
- Loan purpose: Agricultural equipment purchase (income-generating)
- Payment history: 100% on-time for motorcycle loan
- Note: Customer stated income seasonal — peaks Apr-Nov, lower Jan-Mar
"""

def analyze_with_extended_thinking(case: str, thinking_budget: int = 3000):
    """Use extended thinking for complex borderline loan decisions."""
    print(f"Extended Thinking Budget: {thinking_budget} tokens")
    print("-" * 50)

    response = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=5000,
        thinking={"type": "enabled", "budget_tokens": thinking_budget},
        system="""You are LoanBot v5.0 with Extended Thinking enabled.
You specialize in complex, borderline loan decisions that require deep analysis.
Consider: credit score context, income seasonality, DTI nuance, loan purpose,
payment history quality, regional economic factors, and fairness implications.
Output: decision (APPROVE/REVIEW/REJECT), confidence (0-1), key reasoning points,
and recommended loan terms if APPROVE.""",
        messages=[{"role": "user", "content": f"Analyze this loan application:\n{case}"}]
    )

    thinking_tokens = 0
    decision_text = ""

    for block in response.content:
        if block.type == "thinking":
            thinking_tokens = len(block.thinking.split())
            print(f"[Thinking Block: ~{thinking_tokens} words — internal reasoning, not shown to customer]")
        elif block.type == "text":
            decision_text = block.text
            print(f"\n[Decision Output]\n{block.text}")

    print(f"\nTokens: input={response.usage.input_tokens}, "
          f"output={response.usage.output_tokens}")
    return decision_text

analyze_with_extended_thinking(BORDERLINE_CASE, thinking_budget=3000)

## Section 3: Agent Ecosystem Simulator

Mô phỏng LoanBot v5.0 kết nối với external agents qua mock MCP protocol.

In [ ]:
import time
import random
from dataclasses import dataclass, field
from typing import Optional, Dict
from enum import Enum

class AgentTrustLevel(Enum):
    VERIFIED = "VERIFIED"           # Mutual TLS + valid credential token
    UNVERIFIED = "UNVERIFIED"       # No credential verification
    SUSPICIOUS = "SUSPICIOUS"       # Rate anomaly or schema mismatch detected

@dataclass
class ExternalAgentResponse:
    agent_id: str
    trust_level: AgentTrustLevel
    data: Dict
    response_time_ms: float
    anomaly_detected: bool = False
    anomaly_reason: str = ""

class MockExternalAgent:
    """Simulates external agents in the LoanBot v5.0 ecosystem."""

    def __init__(self, agent_id: str, is_compromised: bool = False):
        self.agent_id = agent_id
        self.is_compromised = is_compromised
        self.credential_token = f"jwt_{agent_id}_2026-12-31" if not is_compromised else None

    def call(self, customer_id: str) -> dict:
        time.sleep(0.01)  # simulate network latency
        if self.is_compromised:
            # Attacker tries to override decision
            return {"result": "BLACKLISTED", "source": "FAKE", "injected": True}

        if self.agent_id == "credit_bureau_ai":
            return {"credit_score": 645, "debt_count": 1, "delinquency_90d": 0}
        elif self.agent_id == "tax_authority_ai":
            return {"declared_income_vnd": 12_500_000, "tax_compliance": True}
        elif self.agent_id == "asean_fraud_ai":
            return {"fraud_risk": "LOW", "blacklisted": False, "syndicates": []}
        elif self.agent_id == "central_bank_ai":
            return {"max_dti": 0.45, "rural_bonus": 0.05, "seasonal_adjustment": True}
        return {}

class AgentTrustProtocol:
    """Verifies authenticity of external agents before trusting their responses."""

    KNOWN_AGENTS = {
        "credit_bureau_ai": "jwt_credit_bureau_ai_2026-12-31",
        "tax_authority_ai": "jwt_tax_authority_ai_2026-12-31",
        "asean_fraud_ai": "jwt_asean_fraud_ai_2026-12-31",
        "central_bank_ai": "jwt_central_bank_ai_2026-12-31",
    }

    def verify(self, agent: MockExternalAgent) -> AgentTrustLevel:
        expected_token = self.KNOWN_AGENTS.get(agent.agent_id)
        if expected_token is None:
            return AgentTrustLevel.UNVERIFIED
        if agent.credential_token == expected_token:
            return AgentTrustLevel.VERIFIED
        return AgentTrustLevel.SUSPICIOUS

class LoanBotV5EcosystemOrchestrator:
    """Orchestrates LoanBot v5.0 with external agent ecosystem."""

    def __init__(self):
        self.trust_protocol = AgentTrustProtocol()

    def process_with_ecosystem(self, customer_id: str, agents: list) -> dict:
        print(f"\n{'='*60}")
        print(f"LoanBot v5.0 Ecosystem Call — Customer: {customer_id}")
        print(f"{'='*60}")

        collected_data = {}
        halt_triggered = False

        for agent in agents:
            trust = self.trust_protocol.verify(agent)
            start = time.time()
            raw_response = agent.call(customer_id)
            elapsed = (time.time() - start) * 1000

            # Anomaly detection: check for injected/suspicious data
            anomaly = raw_response.get("injected", False)
            anomaly_reason = "Injected field detected" if anomaly else ""

            ext_response = ExternalAgentResponse(
                agent_id=agent.agent_id,
                trust_level=trust,
                data=raw_response,
                response_time_ms=round(elapsed, 1),
                anomaly_detected=anomaly or trust == AgentTrustLevel.SUSPICIOUS,
                anomaly_reason=anomaly_reason or ("Invalid credential" if trust == AgentTrustLevel.SUSPICIOUS else "")
            )

            status_icon = "✅" if trust == AgentTrustLevel.VERIFIED and not anomaly else "🚨"
            print(f"  {status_icon} {agent.agent_id}: trust={trust.value}, {elapsed:.0f}ms")

            if ext_response.anomaly_detected:
                print(f"     ⚠️  ANOMALY: {ext_response.anomaly_reason} — HALTING, alerting human oversight")
                halt_triggered = True
                break

            if trust == AgentTrustLevel.VERIFIED:
                collected_data[agent.agent_id] = raw_response

        if halt_triggered:
            return {"status": "HALTED", "reason": "External agent anomaly", "action": "Human review required"}

        # Synthesize decision from verified external data
        credit_data = collected_data.get("credit_bureau_ai", {})
        fraud_data = collected_data.get("asean_fraud_ai", {})
        reg_data = collected_data.get("central_bank_ai", {})

        if fraud_data.get("blacklisted"):
            return {"status": "REJECT", "reason": "ASEAN fraud blacklist"}

        score = credit_data.get("credit_score", 0)
        seasonal_adj = reg_data.get("seasonal_adjustment", False)
        adj_threshold = 630 if seasonal_adj else 660  # Central Bank rural bonus

        if score >= adj_threshold:
            decision = "APPROVE"
        elif score >= 580:
            decision = "REVIEW"
        else:
            decision = "REJECT"

        result = {
            "status": decision,
            "credit_score": score,
            "seasonal_adjustment_applied": seasonal_adj,
            "effective_threshold": adj_threshold,
            "external_agents_used": list(collected_data.keys())
        }
        print(f"\n  Decision: {decision} (score={score}, threshold={adj_threshold}, seasonal={seasonal_adj})")
        return result

# Test 1: Normal ecosystem — all agents verified
normal_agents = [
    MockExternalAgent("credit_bureau_ai"),
    MockExternalAgent("tax_authority_ai"),
    MockExternalAgent("asean_fraud_ai"),
    MockExternalAgent("central_bank_ai"),
]
orchestrator = LoanBotV5EcosystemOrchestrator()
result1 = orchestrator.process_with_ecosystem("FC-2024-004", normal_agents)
print(f"\nFinal Result: {result1}")

# Test 2: Compromised agent attack
print("\n" + "─" * 60)
print("ATTACK SCENARIO: Compromised external agent")
attack_agents = [
    MockExternalAgent("credit_bureau_ai"),
    MockExternalAgent("asean_fraud_ai", is_compromised=True),  # Attacker!
    MockExternalAgent("central_bank_ai"),
]
result2 = orchestrator.process_with_ecosystem("FC-2024-004", attack_agents)
print(f"\nFinal Result: {result2}")

## Section 4: Career Path Skill Gap Analyzer

Đánh giá skill gap sau 30 modules và recommend career path phù hợp.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List

@dataclass
class CareerPath:
    name: str
    required_skills: Dict[str, int]  # skill: min score 1-5
    key_modules: List[int]
    vn_salary_range: str
    demand_2028: str

CAREER_PATHS = [
    CareerPath(
        name="AI Engineer / Agent Architect",
        required_skills={"python": 4, "agent_architecture": 4, "multi_agent": 3, "mlops": 3, "security": 3},
        key_modules=[2, 13, 15, 18, 25],
        vn_salary_range="2,000–5,000 USD/month",
        demand_2028="Very High"
    ),
    CareerPath(
        name="AI Product Manager",
        required_skills={"domain_knowledge": 4, "roi_analysis": 4, "hitl_design": 4, "compliance": 3, "python": 2},
        key_modules=[11, 19, 29, 30],
        vn_salary_range="1,500–3,500 USD/month",
        demand_2028="High"
    ),
    CareerPath(
        name="AI Ethics Officer / Governance Lead",
        required_skills={"compliance": 5, "fairness": 5, "domain_knowledge": 4, "agent_architecture": 3, "python": 3},
        key_modules=[9, 19, 28, 20],
        vn_salary_range="1,200–2,500 USD/month",
        demand_2028="High (Emerging)"
    ),
    CareerPath(
        name="MLOps / AI Platform Engineer",
        required_skills={"python": 5, "mlops": 5, "observability": 4, "testing": 4, "cost_optimization": 3},
        key_modules=[8, 10, 12, 20, 27],
        vn_salary_range="1,800–4,000 USD/month",
        demand_2028="High"
    ),
]

class CareerPathAnalyzer:
    def __init__(self, user_skills: Dict[str, int]):
        """user_skills: {skill_name: self_rating_1_to_5}"""
        self.user_skills = user_skills

    def score_fit(self, career: CareerPath) -> tuple:
        total_gap = 0
        gaps = {}
        for skill, required in career.required_skills.items():
            user_level = self.user_skills.get(skill, 0)
            gap = max(0, required - user_level)
            if gap > 0:
                gaps[skill] = (user_level, required)
            total_gap += gap
        fit_score = max(0, 100 - total_gap * 15)  # 15% penalty per gap point
        return fit_score, gaps

    def analyze(self):
        print("=" * 65)
        print("Career Path Fit Analysis — After 30 Modules")
        print("=" * 65)

        results = []
        for career in CAREER_PATHS:
            fit, gaps = self.score_fit(career)
            results.append((fit, career, gaps))

        results.sort(reverse=True, key=lambda x: x[0])

        for rank, (fit, career, gaps) in enumerate(results, 1):
            bar = "█" * (fit // 10) + "░" * (10 - fit // 10)
            icon = "⭐" if rank == 1 else "  "
            print(f"\n{icon} #{rank} {career.name}")
            print(f"    Fit Score: [{bar}] {fit}%")
            print(f"    Salary:    {career.vn_salary_range}")
            print(f"    Demand:    {career.demand_2028}")
            print(f"    Key M's:   {', '.join(f'M{m}' for m in career.key_modules)}")
            if gaps:
                print("    Gaps:")
                for skill, (current, required) in gaps.items():
                    print(f"      • {skill}: {current}/5 → need {required}/5")
            else:
                print("    ✅ No skill gaps — you're ready!")

# === EDIT YOUR SELF-ASSESSMENT HERE ===
# Rate yourself 1 (beginner) to 5 (expert) for each skill:
MY_SKILLS = {
    "python": 3,              # Programming with Python
    "agent_architecture": 4,  # AI agent systems design (from this course)
    "multi_agent": 4,         # Multi-agent orchestration (M26)
    "mlops": 2,               # DevOps for ML models
    "security": 3,            # AI security (M9)
    "domain_knowledge": 4,    # Finance / FinTech domain
    "roi_analysis": 4,        # Business case / ROI (M29)
    "hitl_design": 3,         # Human-in-the-loop design (M11)
    "compliance": 4,          # AI regulation compliance (M19, M28)
    "fairness": 4,            # AI fairness / ethics (M28)
    "observability": 2,       # AgentOps, monitoring (M8)
    "testing": 3,             # Agent QA (M20)
    "cost_optimization": 3,   # Token budgeting (M10, M29)
}

CareerPathAnalyzer(MY_SKILLS).analyze()

## Section 5: Capstone Integration Exercise

### LoanBot v5.0 Full Decision Pipeline

Tích hợp tất cả concepts từ M1–M30: harness + multi-agent + governance + economics + ecosystem.

**Your Task:** Implement `LoanBotV5Pipeline.process()` để orchestrate toàn bộ pipeline:
1. Ecosystem call (external agents)
2. Multi-agent debate (Condorcet, từ M26)
3. Governance check (fairness + policy, từ M28)
4. Economic routing (tier selection, từ M29)
5. XAI explanation (từ M28)

**Expected output:** Dictionary với decision, confidence, explanation, cost, governance_status.

In [ ]:
# TODO: Implement LoanBotV5Pipeline

class LoanBotV5Pipeline:
    """Full capstone pipeline integrating M1-M30 concepts."""

    def process(self, application: dict) -> dict:
        """
        Args:
            application: {
                'customer_id': str,
                'credit_score': int,
                'loan_amount_vnd': int,
                'monthly_income_vnd': int,
                'dti': float,
                'province': str,
                'is_seasonal_worker': bool
            }

        Returns:
            {
                'decision': APPROVE|REVIEW|REJECT,
                'confidence': float,
                'explanation': str,
                'cost_vnd': float,
                'tier': CLEAR|STANDARD|HIGH_VALUE,
                'governance_status': PASS|FLAG|HALT,
                'agents_used': list
            }
        """
        # TODO:
        # Step 1: Determine routing tier (M29 TokenBudgetManager)
        #         CLEAR if score>700 and amount<100M, HIGH_VALUE if amount>500M else STANDARD
        # Step 2: Call external agents (Section 3 above)
        # Step 3: Multi-agent debate with 3 agents (simplified Condorcet from M26)
        #         Use different prompts for Agent1 (conservative), Agent2 (neutral), Agent3 (growth-oriented)
        # Step 4: Check governance (M28): seasonal_adjustment + rural_bonus + DTI check
        # Step 5: Generate explanation (M28 XAI): key factors + counterfactual
        # Step 6: Calculate cost from tier

        raise NotImplementedError("Implement LoanBotV5Pipeline.process()")

# Test case
TEST_APP = {
    "customer_id": "FC-2024-004",
    "credit_score": 645,
    "loan_amount_vnd": 180_000_000,
    "monthly_income_vnd": 12_500_000,
    "dti": 0.42,
    "province": "Nghệ An",
    "is_seasonal_worker": True
}

pipeline = LoanBotV5Pipeline()
# result = pipeline.process(TEST_APP)
# print(result)
print("[TODO] Uncomment the lines above after implementing process()")

In [ ]:
# SOLUTION — Capstone LoanBot v5.0 Pipeline

import anthropic, os
from dataclasses import dataclass
from typing import Optional

class LoanBotV5PipelineSolution:
    """Reference implementation — integrates M1-M30 concepts."""

    TIER_COSTS = {"CLEAR": 54, "STANDARD": 518, "HIGH_VALUE": 1150}  # VNĐ

    def __init__(self):
        self.client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

    # ── Step 1: Tier Routing (M29) ───────────────────────────────────────────
    def _route_tier(self, app: dict) -> str:
        score = app["credit_score"]
        amount = app["loan_amount_vnd"]
        if score >= 700 and amount < 100_000_000:
            return "CLEAR"
        elif amount >= 500_000_000:
            return "HIGH_VALUE"
        return "STANDARD"

    # ── Step 2: Governance Check (M28) ─────────────────────────────────────
    def _governance_check(self, app: dict) -> tuple:
        effective_dti_limit = 0.45 + (0.05 if app.get("is_seasonal_worker") else 0)
        rural_threshold_bonus = 15 if app["province"] not in ["Hà Nội", "TP HCM"] else 0
        effective_approve_threshold = 660 - rural_threshold_bonus

        if app["dti"] > 0.60:
            return "HALT", f"DTI {app['dti']:.0%} exceeds hard cap 60%"
        if app["dti"] > effective_dti_limit:
            return "FLAG", f"DTI {app['dti']:.0%} above adjusted limit {effective_dti_limit:.0%}"
        return "PASS", f"Approved threshold: {effective_approve_threshold} (rural -{rural_threshold_bonus}pts, seasonal DTI +5%)"

    # ── Step 3: Multi-Agent Debate (M26 simplified) ─────────────────────────
    def _debate(self, app: dict) -> tuple:
        prompts = [
            ("conservative_agent", "You are a risk-conservative analyst. Focus on DTI, borderline score risks."),
            ("neutral_agent", "You are a balanced credit analyst. Weigh all factors objectively."),
            ("growth_agent", "You are a growth-oriented analyst. Emphasize income quality, purpose, payment history."),
        ]

        votes = {"APPROVE": 0, "REVIEW": 0, "REJECT": 0}
        print("  [Multi-Agent Debate]")

        for agent_name, system_prompt in prompts:
            resp = self.client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=100,
                system=system_prompt + "\nRespond with exactly one word: APPROVE, REVIEW, or REJECT.",
                messages=[{"role": "user", "content":
                    f"Credit score: {app['credit_score']}, DTI: {app['dti']:.0%}, "
                    f"income: {app['monthly_income_vnd']:,} VNĐ, seasonal: {app['is_seasonal_worker']}, "
                    f"amount: {app['loan_amount_vnd']:,} VNĐ. Decision?"
                }]
            )
            vote = resp.content[0].text.strip().upper()
            if vote not in votes:
                vote = "REVIEW"
            votes[vote] += 1
            print(f"    {agent_name}: {vote}")

        majority = max(votes, key=votes.get)
        confidence = votes[majority] / 3.0
        print(f"    Condorcet winner: {majority} ({votes[majority]}/3 votes)")
        return majority, confidence

    # ── Step 4: XAI Explanation (M28) ───────────────────────────────────────
    def _explain(self, app: dict, decision: str, gov_note: str) -> str:
        factors = []
        score = app["credit_score"]
        if score >= 660:
            factors.append(f"Credit score {score} đạt ngưỡng chuẩn (≥660)")
        elif score >= 645:
            factors.append(f"Credit score {score} được hưởng điều chỉnh địa phương (ngưỡng giảm về 645)")
        if app.get("is_seasonal_worker"):
            factors.append("Thu nhập theo mùa được tính với DTI linh hoạt (+5%)")
        counterfactual = ""
        if decision == "REVIEW":
            counterfactual = f" Nếu điểm tín dụng ≥660 và DTI ≤40%, khoản vay sẽ được DUYỆT TỰ ĐỘNG."
        return f"Quyết định: {decision}. Yếu tố chính: {'; '.join(factors)}.{counterfactual} ({gov_note})"

    # ── Main Pipeline ────────────────────────────────────────────────────────
    def process(self, app: dict) -> dict:
        print(f"\n{'='*60}")
        print(f"LoanBot v5.0 — Full Pipeline — {app['customer_id']}")
        print(f"{'='*60}")

        # Step 1: Tier
        tier = self._route_tier(app)
        cost = self.TIER_COSTS[tier]
        print(f"  Tier: {tier} ({cost} VNĐ/case)")

        # Step 2: Governance
        gov_status, gov_note = self._governance_check(app)
        print(f"  Governance: {gov_status} — {gov_note}")
        if gov_status == "HALT":
            return {"decision": "REJECT", "confidence": 1.0, "explanation": gov_note,
                    "cost_vnd": cost, "tier": tier, "governance_status": "HALT", "agents_used": []}

        # Step 3: Debate (skipped for CLEAR tier)
        if tier == "CLEAR":
            decision = "APPROVE"
            confidence = 0.95
            print("  Debate: Skipped (CLEAR tier — direct approve)")
        else:
            decision, confidence = self._debate(app)

        # Step 4: Explanation
        explanation = self._explain(app, decision, gov_note)
        print(f"  Explanation: {explanation[:80]}...")

        return {
            "decision": decision,
            "confidence": round(confidence, 2),
            "explanation": explanation,
            "cost_vnd": cost,
            "tier": tier,
            "governance_status": gov_status,
            "agents_used": ["credit_bureau_ai", "asean_fraud_ai", "central_bank_ai"]
        }

# Run capstone pipeline
capstone = LoanBotV5PipelineSolution()
final_result = capstone.process(TEST_APP)

print("\n" + "=" * 60)
print("FINAL RESULT")
print("=" * 60)
for k, v in final_result.items():
    print(f"  {k:25s}: {v}")

print("\n🎓 Capstone complete — bạn đã tích hợp thành công M1-M30!")